# DDL-Diffusion

Latent text-to-image diffusion built from scratch: Stable Diffusion's VAE, a CLIP text
encoder, and a hand-written U-Net, trained on Multi-Modal CelebA-HQ. `RUNS.md` has the
training log.

Runs 1-4 used 833 captioned Pokemon and stalled at a held-out v-loss of ~0.57. The cause was
measured, not guessed: 750 distinct caption-image pairs is not enough to learn a
caption-to-layout mapping, and no amount of architecture work moved it. CelebA-HQ has 30,000
images with 10 captions each, keeps the single-centred-subject structure that makes the
target easy, and multiplies distinct images by 36x. See "Dataset choice for run 5" in
`RUNS.md`.

Run top to bottom. The encoding cells write four `.pt` files, so once they have run you can
restart the kernel and go straight to training.

## Setup

Dependency check, imports, seed, dataset.

In [ ]:
# Environment setup. Written for a rented single-GPU Linux box (vast.ai), and works the same
# on any local machine with a CUDA GPU.
#
# Nothing needs uploading: the dataset, the VAE and CLIP all download from the HuggingFace Hub
# at runtime, and `Ryan-sjtu/celebahq-caption` is public and ungated so no token is needed. The
# Hub cache lives under ~/.cache/huggingface on the instance disk, so a fresh instance
# re-downloads 2.76 GB. Everything this notebook writes (the .pt files and checkpoints/) lands
# next to it in the working directory, which on an ephemeral instance means copying the
# checkpoints off before you stop it.
import os
import subprocess
import sys
from pathlib import Path

# A fresh container ships torch but usually not these.
try:
    import diffusers, transformers, datasets  # noqa: F401
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "datasets"],
        check=True,
    )

# The .py modules must sit beside the notebook: scheduler, unet, train, sample, precompute.
if not Path("scheduler.py").is_file():
    raise RuntimeError(
        f"no scheduler.py in {os.getcwd()} — run the notebook from the repo's code/ directory"
    )

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("cwd  :", os.getcwd())

In [ ]:
import torch
import datasets
import numpy as np
import matplotlib.pyplot as plt
from diffusers import AutoencoderKL
from tqdm import tqdm
from transformers import CLIPTokenizer, CLIPTextModel

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
device

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

In [ ]:
# Verified 2026-08-31 against the dataset viewer: 30,000 rows, columns `image` and `text`,
# 2.76 GB parquet. One caption per image, not the ten the MM-CelebA paper describes, because
# IIGROUP/MM-CelebA-HQ-Dataset returns 401 and that text cannot be fetched and joined.
#
# The captions are BLIP-style free text rather than attribute templates:
#   "a photography of a woman with a very long blond hair"
#   "a photography of a young man with a necklace and a black shirt"
# Nearly all open with "a photography of", so inference prompts should carry that prefix.
# See `Run 5 config` in RUNS.md.
DATASET_ID = "Ryan-sjtu/celebahq-caption"
SPLIT = "train"
IMAGE_COL = "image"
CAPTION_COL = "text"

raw = datasets.load_dataset(DATASET_ID)
print(raw)

DATA = raw[SPLIT]
cols = list(DATA.features)
assert IMAGE_COL in cols, f"no {IMAGE_COL!r} column; columns are {cols}"
assert CAPTION_COL in cols, f"no {CAPTION_COL!r} column; columns are {cols}"

# A caption cell holds either one string or a list of them. Normalize to a list either way.
caption_lists = [[c] if isinstance(c, str) else [str(x) for x in c]
                 for c in DATA[CAPTION_COL]]
N_IMAGES = len(DATA)
CAPS_MIN = min(len(c) for c in caption_lists)
CAPS_MAX = max(len(c) for c in caption_lists)

sample = DATA[0]
print(f"\nimages         : {N_IMAGES}")
print(f"captions/image : min {CAPS_MIN}  max {CAPS_MAX}")
print(f"first captions : {caption_lists[0][:3]}")
if CAPS_MIN == 1:
    print("\nOnly one caption per image. If this dataset stores one ROW per caption then the\n"
          "images are duplicated and N_IMAGES is wrong. Group by image before continuing.")

## Images → latents

**One-off.** Encode every crop and flip through the VAE, normalize each channel to zero mean
and unit standard deviation, and save `latents.pt`.

On a rerun, skip the encode cell below (the slow one, ~1.5 h at 30k images) and the decode
check after it. Everything downstream reads the saved `.pt` files instead.

In [ ]:
RESOLUTION = 256
LATENT_SIZE = RESOLUTION // 8 # like SD
VAE_SCALE = 0.18215
CROPS = 1 # 1 = centre crop only. Run 3: 4 crops left 59.9% of the target unpredictable
CROP_SCALE = (0.8, 1.0)
HFLIP = True
CAPTIONS_PER_IMAGE = 1 # real captions kept per image; drives the size of embeddings.pt.
                       # This dataset ships 1, so N_CAPS clamps to 1 either way. At 30k images
                       # each caption per image costs ~1.5 GB of RAM, which matters on Flickr30k
MAX_LENGTH = 32 # CLIP token budget. SD uses 77; cell 13 measures the real p99
VIEW_COND = True # micro-condition on the crop box + flip flag

from precompute import build_preprocess, crop_with_params, VIEW_DIM, CANONICAL_VIEW

preprocess = build_preprocess(RESOLUTION, crop=False)          # canonical view, for display
test_image = preprocess(sample[IMAGE_COL])
print(test_image.shape, "-> latents will be", (4, LATENT_SIZE, LATENT_SIZE))
print(f"view conditioning: {VIEW_COND}   view_dim {VIEW_DIM}   canonical {CANONICAL_VIEW}")

n_rows = N_IMAGES * CROPS * (2 if HFLIP else 1)
lat_gb = n_rows * 4 * LATENT_SIZE ** 2 * 4 / 1e9
emb_gb = N_IMAGES * CAPTIONS_PER_IMAGE * MAX_LENGTH * 768 * 2 / 1e9
print(f"planned latents  : {N_IMAGES} x {CROPS} crops x {2 if HFLIP else 1} flips "
      f"= {n_rows}   ({lat_gb:.2f} GB fp32)")
print(f"planned captions : {N_IMAGES} x {CAPTIONS_PER_IMAGE} = {N_IMAGES * CAPTIONS_PER_IMAGE}"
      f"   ({emb_gb:.2f} GB fp16)")
if CAPTIONS_PER_IMAGE > CAPS_MIN:
    print(f"  note: some images have only {CAPS_MIN} captions, theirs will be cycled")
if emb_gb > 3:
    print("  embeddings.pt is large: lower CAPTIONS_PER_IMAGE or MAX_LENGTH")

# The crops, with the view vector recorded for each.
fig, axes = plt.subplots(1, CROPS, figsize=(3.6 * CROPS, 3.6), squeeze=False)
for k in range(CROPS):
    img, view = crop_with_params(sample[IMAGE_COL], RESOLUTION, crop=k > 0, scale=CROP_SCALE)
    axes[0][k].imshow(((img + 1) / 2).permute(1, 2, 0).numpy())
    axes[0][k].set_title(("crop 0 (centre)" if k == 0 else f"random crop {k}") +
                         "\n" + str([round(v, 2) for v in view]), fontsize=8)
    axes[0][k].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
vae.requires_grad_(False)

In [ ]:
# Encode the whole dataset -> latents.pt + latent_stats.pt
#
# One VAE pass per (crop, flip) combination, plus four index arrays:
#   group_ids   : source image. make_split splits on this, so all crops and flips of one
#                 image stay on the same side of the train/val boundary.
#   crop_ids    : which crop. Validation uses crop 0 only.
#   flip_ids    : mirrored or not.
#   view_params : (N, 5) = [top, left, h, w, flip], normalized against the source image.
#                 Passed to the U-Net so it knows the framing instead of averaging over it.
from precompute import normalize_per_channel

def encode_pass(crop, flip, desc):
    lats, views = [], []
    for i in tqdm(range(0, N_IMAGES, 16), desc=desc, leave=False):
        pairs = [crop_with_params(im, RESOLUTION, crop=crop, scale=CROP_SCALE, flip=flip)
                 for im in DATA[i : i + 16][IMAGE_COL]]
        imgs = torch.stack([p[0] for p in pairs]).to(device)
        views.extend(p[1] for p in pairs)
        with torch.no_grad():
            lat = vae.encode(imgs).latent_dist.mode() * VAE_SCALE
        lats.append(lat.cpu())
    return torch.cat(lats, dim=0), torch.tensor(views, dtype=torch.float32)

flips = (False, True) if HFLIP else (False,)

chunks, vchunks, group_ids, crop_ids, flip_ids = [], [], [], [], []
for crop_i in range(CROPS):
    torch.manual_seed(SEED + crop_i)          # reproducible crop geometry
    for flip in flips:
        tag = f"crop {crop_i}{' flip' if flip else ''}"
        lat, vw = encode_pass(crop_i > 0, flip, tag)
        chunks.append(lat); vchunks.append(vw)
        group_ids.append(torch.arange(N_IMAGES))
        crop_ids.append(torch.full((N_IMAGES,), crop_i))
        flip_ids.append(torch.full((N_IMAGES,), int(flip)))
        print(f"  {tag:14s} done   view e.g. {[round(v, 2) for v in vw[0].tolist()]}")

raw_latents = torch.cat(chunks, dim=0)
view_params = torch.cat(vchunks, dim=0)
group_ids = torch.cat(group_ids).long()
crop_ids = torch.cat(crop_ids).long()
flip_ids = torch.cat(flip_ids).long()

print(f"\nraw {tuple(raw_latents.shape)}   views {tuple(view_params.shape)}")
print("  per-chan mean:", [round(v, 3) for v in raw_latents.mean(dim=(0, 2, 3)).tolist()])
print("  per-chan std :", [round(v, 3) for v in raw_latents.std(dim=(0, 2, 3)).tolist()])

latents_tensor, LAT_MEAN, LAT_STD = normalize_per_channel(raw_latents)
print("\nnormalized")
print("  per-chan mean:", [round(v, 4) for v in latents_tensor.mean(dim=(0, 2, 3)).tolist()], "(want ~0)")
print("  per-chan std :", [round(v, 4) for v in latents_tensor.std(dim=(0, 2, 3)).tolist()], "(want ~1)")
print(f"  overall mean {latents_tensor.mean():+.5f}  std {latents_tensor.std():.5f}  "
      f"abs max {latents_tensor.abs().max():.2f}")

_X = latents_tensor.flatten(1).double()
_s = torch.zeros(N_IMAGES, _X.shape[1], dtype=torch.float64)
_c = torch.zeros(N_IMAGES, dtype=torch.float64)
_s.index_add_(0, group_ids, _X); _c.index_add_(0, group_ids, torch.ones(len(_X), dtype=torch.float64))
_gm = _s / _c[:, None]
_w = ((_X - _gm[group_ids]) ** 2).mean().item()
_t = ((_X - _X.mean(0)) ** 2).mean().item()
print(f"\nview noise (variance not predictable from the caption): {_w/_t:.1%}")

del _X, _s, _c, _gm

torch.save(latents_tensor, "latents.pt")
del raw_latents, chunks
print("\nsaved latents.pt")

In [ ]:
from sample import latents_to_images

# Row layout is [crop0, crop0-flip, crop1, crop1-flip, ...], so image 0 under each
# (crop, flip) pass sits at pass_index * N_IMAGES.
n_passes = CROPS * (2 if HFLIP else 1)
picks = [p * N_IMAGES for p in range(min(n_passes, 4))]

fig, axes = plt.subplots(1, len(picks) + 1, figsize=(3.2 * (len(picks) + 1), 3.8))
for ax, row in zip(axes, picks):
    img = latents_to_images(latents_tensor[row : row + 1].to(device),
                            vae, lat_mean=LAT_MEAN, lat_std=LAT_STD)[0].cpu()
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"row {row}  crop {int(crop_ids[row])}"
                 f"{' flip' if int(flip_ids[row]) else ''}", fontsize=9)

bad = latents_to_images(latents_tensor[0:1].to(device), vae)[0].cpu()
axes[-1].imshow(bad.permute(1, 2, 0).numpy())
axes[-1].set_title("lat_mean / lat_std NOT undone", fontsize=9)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Captions → embeddings

**One-off.** Encode `CAPTIONS_PER_IMAGE` of each image's real captions with CLIP, plus the
empty string that CFG needs, into `embeddings.pt` and `uncond_embedding.pt`.

The cell that loads the tokenizer and text encoder is cheap and worth re-running; the two
encode cells after it are not, since their output is already on disk.

In [ ]:
CLIP_NAME = "openai/clip-vit-large-patch14"

tokenizer = CLIPTokenizer.from_pretrained(CLIP_NAME)
text_encoder = CLIPTextModel.from_pretrained(CLIP_NAME).to(device)
text_encoder.eval()
text_encoder.requires_grad_(False)

In [ ]:
# Encode captions -> embeddings.pt, and write latent_stats.pt
# Captions are stored once per (image, caption) and joined to latents by group id rather
# than duplicated per crop. Saved fp16; LatentCaptionDataset casts back to fp32.

from precompute import build_multi_captions

# Clamped: cycling a caption would store byte-identical embeddings and waste the RAM.
N_CAPS = min(CAPTIONS_PER_IMAGE, CAPS_MIN)
if N_CAPS != CAPTIONS_PER_IMAGE:
    print(f"CAPTIONS_PER_IMAGE {CAPTIONS_PER_IMAGE} clamped to {N_CAPS} (the per-image minimum)")

flat_captions, caption_groups, n_cycled = build_multi_captions(caption_lists, N_CAPS)
print(f"{N_IMAGES} images x {N_CAPS} captions = {len(flat_captions)} captions")
assert n_cycled == 0, "clamping should have prevented cycling"
print(f"\nimage 0's captions:")
for c in flat_captions[:N_CAPS]:
    print(f"  {c}")

# MAX_LENGTH must not be truncating anything informative.
tok_lens = [len(tokenizer(c).input_ids) for c in flat_captions]
p99 = int(np.percentile(tok_lens, 99))
print(f"\ntoken lengths: mean {np.mean(tok_lens):.1f}  p99 {p99}  max {max(tok_lens)}"
      f"   MAX_LENGTH {MAX_LENGTH}")
if p99 > MAX_LENGTH:
    print(f"  WARNING: p99 exceeds MAX_LENGTH, raise it to at least {p99}")

all_input_ids = tokenizer(
    flat_captions,
    padding="max_length",
    max_length=MAX_LENGTH,
    truncation=True,
    return_tensors="pt",
).input_ids

all_embeddings = []
for i in tqdm(range(0, len(flat_captions), 64), desc="CLIP encode"):
    with torch.no_grad():
        emb = text_encoder(all_input_ids[i : i + 64].to(device)).last_hidden_state
    all_embeddings.append(emb.half().cpu())

embeddings_tensor = torch.cat(all_embeddings, dim=0)
print(f"\nembeddings {tuple(embeddings_tensor.shape)} {embeddings_tensor.dtype}   "
      f"{embeddings_tensor.numel() * 2 / 1e9:.2f} GB")
torch.save(embeddings_tensor, "embeddings.pt")

torch.save({
    "mean": LAT_MEAN, "std": LAT_STD, "scale": VAE_SCALE,
    "resolution": RESOLUTION, "latent_size": latents_tensor.shape[-1],
    "latent_channels": latents_tensor.shape[1], "n_images": N_IMAGES,
    "crops": CROPS, "crop_scale": CROP_SCALE, "hflip": HFLIP,
    "captions_per_image": N_CAPS, "max_length": MAX_LENGTH,
    "dataset_id": DATASET_ID, "normalized": True,
    "group_ids": group_ids, "crop_ids": crop_ids, "flip_ids": flip_ids,
    "caption_groups": caption_groups, "seed": SEED,
    "view_params": view_params, "view_dim": VIEW_DIM,
    "canonical_view": torch.tensor(CANONICAL_VIEW), "view_cond": VIEW_COND,
}, "latent_stats.pt")
print("saved latent_stats.pt (index arrays + view params + normalization stats)")

In [ ]:
# Encode the "" -> uncond_embedding.pt
# Classifier-Free Guidance (CFG) during training and inference.
uncond_ids = tokenizer(
    "",
    padding="max_length",
    max_length=MAX_LENGTH,
    truncation=True,
    return_tensors="pt",
).input_ids.to(device)

with torch.no_grad():
    uncond_embedding = text_encoder(uncond_ids).last_hidden_state

print(uncond_embedding.shape)   # (1, MAX_LENGTH, 768)
torch.save(uncond_embedding.cpu(), "uncond_embedding.pt")

## Noise schedule

Confirm the schedule reaches zero terminal SNR, then look at what the forward process does
to a real latent.

In [ ]:
from scheduler import NoiseScheduler

scheduler = NoiseScheduler(zero_terminal_snr=True, prediction_type="v").to(device)

rows = [
    ("alphas_cumprod[0]",        scheduler.alphas_cumprod[0].item(),       "want ~1.0"),
    ("alphas_cumprod[-1]",       scheduler.alphas_cumprod[-1].item(),      "want exactly 0"),
    ("sqrt(alphas_cumprod[-1])", scheduler.sqrt_alphas_cumprod[-1].item(), "0 = no signal survives to t=T"),
    ("betas[0]",                 scheduler.betas[0].item(),                ""),
    ("betas[-1]",                scheduler.betas[-1].item(),               "1.0 is expected here"),
]
for name, val, note in rows:
    print(f"{name:26s} {val:12.7f}   {note}")

assert scheduler.alphas_cumprod[-1].item() == 0.0, "terminal SNR is not zero"
print("\nx_T is pure noise, which is what inference starts from.")

In [ ]:
x_0 = latents_tensor[0:1].to(device)

noise = torch.randn_like(x_0)
timesteps_to_show = [0, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(3.2 * len(timesteps_to_show), 3.8))
for ax, t_val in zip(axes, timesteps_to_show):
    t = torch.full((1,), t_val, device=device, dtype=torch.long)
    x_t = scheduler.q_sample(x_0, t, noise)
    img = latents_to_images(x_t, vae, lat_mean=LAT_MEAN, lat_std=LAT_STD)[0].cpu()
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"t = {t_val}\nsqrt(ab) = {scheduler.sqrt_alphas_cumprod[t_val].item():.4f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Model & training

Hold out 10% of the images, build the U-Net, and train.

**Rerunning after a kernel restart:** run the three setup cells at the top, then come
straight here. The first cell below reloads the four `.pt` files, and the training cell
loads the VAE and CLIP on demand for its previews, so nothing above needs to run again.

In [ ]:
from torch.utils.data import DataLoader, Subset
from train import make_split, LatentCaptionDataset

# because of kernel restarts
latents_tensor = torch.load("latents.pt")             # per-channel normalized
embeddings_tensor = torch.load("embeddings.pt")       # (n_img * caps, MAX_LENGTH, 768) fp16
uncond_embedding = torch.load("uncond_embedding.pt")  # (1, MAX_LENGTH, 768) CLIP-encoded ""
stats = torch.load("latent_stats.pt")

# From latent_stats.pt, so these always match the encoded data.
LAT_MEAN, LAT_STD = stats["mean"], stats["std"]
LATENT_SIZE = stats["latent_size"]
LATENT_SHAPE = (stats["latent_channels"], LATENT_SIZE, LATENT_SIZE)
N_IMAGES = stats["n_images"]
CAPTIONS_PER_IMAGE = stats["captions_per_image"]
MAX_LENGTH = stats["max_length"]
group_ids, crop_ids = stats["group_ids"], stats["crop_ids"]
caption_groups = stats["caption_groups"]
VIEW_COND = bool(stats.get("view_cond", False))
view_params = stats["view_params"] if VIEW_COND else None
CANON = stats["canonical_view"]

print("latents   :", tuple(latents_tensor.shape), latents_tensor.dtype)
print("embeddings:", tuple(embeddings_tensor.shape), embeddings_tensor.dtype,
      f"({CAPTIONS_PER_IMAGE} captions/image)")
print("view cond :", VIEW_COND, "" if not VIEW_COND else f"dim {stats['view_dim']}  canonical {CANON.tolist()}")
print("per-chan mean:", [round(v, 4) for v in latents_tensor.mean(dim=(0, 2, 3)).tolist()], "(want ~0)")
print("per-chan std :", [round(v, 4) for v in latents_tensor.std(dim=(0, 2, 3)).tolist()], "(want ~1)")

# Held-out split on groups, so every crop and flip of one image stays together.
VAL_FRAC = 0.1
train_idx, val_idx = make_split(group_ids, val_frac=VAL_FRAC, seed=0)

# The preview and evaluation cells sample these and label them training images.
# Asserted here: if VAL_FRAC or the seed changes one could land in validation, which would
# invert the memorization test below.
SHOW_IMAGES = [1, 2, 3, 4]
assert set(SHOW_IMAGES) <= set(group_ids[train_idx].tolist()), \
    f"{SHOW_IMAGES} are not all training images — pick other ids or change VAL_FRAC"

full_dataset = LatentCaptionDataset(
    latents_tensor, group_ids, embeddings_tensor, caption_groups,
    view_params=view_params, sample_variant=True,
)
caption_table = full_dataset.caption_table      # (n_images, n_variants) -> embedding row


def caption_emb(image_ids, variant=0):
    """(B, MAX_LENGTH, 768) for a list of source image ids. Stored fp16, returned fp32."""
    idx = caption_table[torch.as_tensor(image_ids, dtype=torch.long), variant]
    return embeddings_tensor[idx].float()


def caption_emb_for_rows(rows, variant=0):
    """(B, MAX_LENGTH, 768) for latent row indices."""
    return caption_emb(group_ids[torch.as_tensor(rows, dtype=torch.long)], variant)


def views_for_rows(rows):
    """(B, view_dim) view params for latent row indices, or None if unused."""
    return None if view_params is None else view_params[torch.as_tensor(rows, dtype=torch.long)]


def canonical_views(n):
    """(n, view_dim) the full-frame unflipped view, used at inference."""
    return None if not VIEW_COND else CANON.unsqueeze(0).repeat(n, 1)


BATCH_SIZE = 32
train_loader = DataLoader(
    Subset(full_dataset, train_idx.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,                 # latents are already RAM tensors, so workers add only overhead
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

val_eval_rows = val_idx[crop_ids[val_idx] == 0]
val_x0, val_ctx = latents_tensor[val_eval_rows], caption_emb_for_rows(val_eval_rows)
val_view = views_for_rows(val_eval_rows)

tr_eval_pool = train_idx[crop_ids[train_idx] == 0]
_g = torch.Generator().manual_seed(0)
tr_eval_rows = tr_eval_pool[torch.randperm(len(tr_eval_pool), generator=_g)[: len(val_eval_rows)]]
tr_x0, tr_ctx = latents_tensor[tr_eval_rows], caption_emb_for_rows(tr_eval_rows)
tr_view = views_for_rows(tr_eval_rows)

uncond_embedding = uncond_embedding.to(device)
print(f"\nimages     : {len(torch.unique(group_ids[train_idx]))} train / {len(torch.unique(group_ids[val_idx]))} val")
print(f"latent rows: {len(train_idx)} train / {len(val_idx)} val")
print(f"eval slices: {len(tr_eval_rows)} train / {len(val_eval_rows)} val  (crop 0 only)")
print(f"batches/epoch: {len(train_loader)}")

In [ ]:
from unet import UNet
from scheduler import NoiseScheduler
from train import build_ema, make_optimizer, make_lr_schedule, pick_amp_dtype

# Model shape
BASE_CHANNELS = 192                 # 129M params. Run 5 at 128 flattened without overfitting
NUM_RES_BLOCKS = 2
TOP_SELF_ATTN = False               # 50% of the forward pass at 64x64; SD has none there either
DROPOUT = 0.0                       # was 0.1: a run-3 patch for 750 images
VIEW_DIM_USED = stats["view_dim"] if VIEW_COND else 0

# TF32 on Ampere and newer. Free speedup, plan.md step 8.
torch.set_float32_matmul_precision("high")

# Optimization
# 1,687 batches/epoch: 30k images x 2 flips, 90% train, batch 32. Run 5 measured 30-40 s
# per epoch on a 5090 at base128, so base192 for 60 epochs is roughly 1.5-2 h. The cosine
# LR anneals to 0 at NUM_EPOCHS, so set this before starting rather than stopping early.
NUM_EPOCHS = 60
WARMUP_STEPS = 500
LR = 2e-4
WEIGHT_DECAY = 0.01               # was 0.05, same reason
EMA_DECAY = 0.999
EMA_WARMUP = True
CFG_DROPOUT = 0.15
GRAD_CLIP = 1.0
MIN_SNR_GAMMA = None

amp_dtype, AMP = pick_amp_dtype("auto", device=device)

TOTAL_STEPS = NUM_EPOCHS * len(train_loader)

model = UNet(base_channels=BASE_CHANNELS, num_res_blocks=NUM_RES_BLOCKS,
             top_self_attn=TOP_SELF_ATTN, dropout=DROPOUT, view_dim=VIEW_DIM_USED).to(device)
noise_scheduler = NoiseScheduler(zero_terminal_snr=True, prediction_type="v").to(device)
ema_model = build_ema(model)
optimizer = make_optimizer(model, lr=LR, weight_decay=WEIGHT_DECAY)
lr_scheduler = make_lr_schedule(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS,
)

scaler = torch.amp.GradScaler("cuda") if (amp_dtype is torch.float16 and device == "cuda") else None

MODEL_CONFIG = {
    "base_channels": BASE_CHANNELS, "num_res_blocks": NUM_RES_BLOCKS,
    "top_self_attn": TOP_SELF_ATTN, "dropout": DROPOUT, "view_dim": VIEW_DIM_USED,
    "prediction_type": "v", "zero_terminal_snr": True,
    "latent_shape": LATENT_SHAPE, "resolution": stats["resolution"],
    "min_snr_gamma": MIN_SNR_GAMMA, "batch_size": BATCH_SIZE, "lr": LR,
    "weight_decay": WEIGHT_DECAY, "cfg_dropout": CFG_DROPOUT, "ema_decay": EMA_DECAY,
    "crops": stats["crops"], "hflip": stats["hflip"],
    "captions_per_image": stats["captions_per_image"],
    "max_length": stats["max_length"], "dataset_id": stats.get("dataset_id"),
    "num_epochs": NUM_EPOCHS, "warmup_steps": WARMUP_STEPS,
    "grad_clip": GRAD_CLIP, "seed": SEED, "view_cond": VIEW_COND,
}

print(f"Trainable params     : {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps          : {TOTAL_STEPS:,}  ({NUM_EPOCHS} epochs x {len(train_loader)} batches)")
print(f"Sample presentations : {TOTAL_STEPS * BATCH_SIZE:,}")
print(f"Peak LR              : {LR}   weight decay {WEIGHT_DECAY}   dropout {DROPOUT}")
print(f"EMA decay            : {EMA_DECAY}")
print(f"View conditioning    : {VIEW_COND} (dim {VIEW_DIM_USED})")
print(f"AMP                  : {AMP}   (GradScaler: {scaler is not None})")
print(f"Params per data scalar: "
      f"{sum(p.numel() for p in model.parameters()) / (len(train_idx) * LATENT_SHAPE[0] * LATENT_SIZE ** 2):.2f}x")

In [ ]:
# Forward-only sanity check: no optimizer step, no EMA update, nothing to reset before
# the real run. Checks that a batch runs through the model and that the loss starts near
# 1.0, the predict-zero baseline.
from train import diffusion_loss

x_0, context, view = next(iter(train_loader))
x_0, context = x_0.to(device), context.to(device)
view = view.to(device) if VIEW_COND else None

model.eval()
with torch.no_grad():
    t = torch.randint(0, noise_scheduler.T, (x_0.shape[0],), device=device)
    noise = torch.randn_like(x_0)
    loss = diffusion_loss(model, noise_scheduler, x_0, context, t, noise,
                          min_snr_gamma=MIN_SNR_GAMMA, view=view).item()
model.train()

print("batch      :", f"x_0 {tuple(x_0.shape)}  context {tuple(context.shape)}",
      f"view {tuple(view.shape)}" if view is not None else "view None")
if view is not None:
    print(f"view row 0 : {[round(v, 2) for v in view[0].tolist()]}")
print(f"loss       : {loss:.4f}   (want ~1.0 — the predict-zero baseline)")

In [ ]:
from tqdm.auto import tqdm
from train import train_step, validation_loss, save_checkpoint, load_checkpoint
from sample import sample_to_image

CHECKPOINT_DIR = "checkpoints"
LOG_EVERY = 50
VAL_EVERY = 2
RESUME_PATH = None               # path to last.pt, None to start fresh
KEEP_BEST = True
MILESTONES = [15, 30, 45, 60]
FULL_EVERY = 5                   # full (resumable) last.pt every 5 epochs. Only a full
                                 # checkpoint can resume: milestones and best.pt are slim and
                                 # carry no optimizer state. Cheap insurance on a rented box

PREVIEW = True
PREVIEW_EVERY = 10
PREVIEW_STEPS = 50
PREVIEW_CFG = 2.0

# The preview decodes latents, so it needs the VAE. It is already in memory when the notebook
# runs top to bottom, but not in a fresh session that skips the precompute cells and loads the
# .pt files instead. Load it on demand rather than dying at the first preview, an hour in.
if PREVIEW and "vae" not in globals():
    from diffusers import AutoencoderKL

    vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
    vae.eval()
    vae.requires_grad_(False)

# Same for CLIP, which the preview needs to encode PREVIEW_NOVEL. Together the two cost
# about 800 MB of VRAM, which is free at 32x32 latents.
if PREVIEW and "tokenizer" not in globals():
    from transformers import CLIPTokenizer, CLIPTextModel

    CLIP_NAME = "openai/clip-vit-large-patch14"
    tokenizer = CLIPTokenizer.from_pretrained(CLIP_NAME)
    text_encoder = CLIPTextModel.from_pretrained(CLIP_NAME).to(device)
    text_encoder.eval()
    text_encoder.requires_grad_(False)
PREVIEW_CFG_RESCALE = 0.7
PREVIEW_IMAGES = SHOW_IMAGES[:3]
PREVIEW_NOVEL = "a smiling young person with long blond hair and high cheekbones"
PREVIEW_DIR = "previews"


@torch.no_grad()
def _clip_encode(prompts):
    """CLIP-encode prompts, same pipeline as the training captions."""
    ids = tokenizer(prompts, padding="max_length", max_length=MAX_LENGTH,
                    truncation=True, return_tensors="pt").input_ids.to(device)
    return text_encoder(ids).last_hidden_state.cpu()

_prev_cond = torch.cat([caption_emb(PREVIEW_IMAGES), _clip_encode([PREVIEW_NOVEL])], dim=0)
_prev_titles = [f"train #{i}" for i in PREVIEW_IMAGES] + ["UNSEEN"]


def make_preview(epoch: int) -> None:
    """DDIM samples from the EMA model: save a PNG strip and show it inline."""
    from torchvision.utils import make_grid, save_image

    g = torch.Generator(device=device).manual_seed(SEED)
    imgs = sample_to_image(
        model=ema_model,
        scheduler=noise_scheduler,
        vae=vae,
        cond_emb=_prev_cond.to(device),
        uncond_emb=uncond_embedding,
        method="ddim",
        guidance_scale=PREVIEW_CFG,
        guidance_rescale=PREVIEW_CFG_RESCALE,
        num_steps=PREVIEW_STEPS,
        latent_shape=LATENT_SHAPE,
        lat_mean=LAT_MEAN,
        lat_std=LAT_STD,
        view=canonical_views(_prev_cond.shape[0]).to(device) if VIEW_COND else None,
        generator=g,
    ).cpu()

    out_path = Path(PREVIEW_DIR) / f"epoch_{epoch}.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    grid = make_grid(imgs, nrow=imgs.shape[0])
    save_image(grid, str(out_path))

    plt.figure(figsize=(3.2 * imgs.shape[0], 3.6))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title(f"epoch {epoch}   {' | '.join(_prev_titles)}   (CFG {PREVIEW_CFG})")
    plt.axis("off")
    plt.show()


start_step = 0
start_epoch = 0
loss_history: list[float] = []
val_history: list[tuple[int, float, float]] = []   # (step, train, val)
best_val = float("inf")
best_epoch = -1

if RESUME_PATH is not None:
    start_step, start_epoch, loss_history = load_checkpoint(
        RESUME_PATH, model, ema_model, optimizer, lr_scheduler,
        map_location=device, scaler=scaler,
    )
    model.to(device); ema_model.to(device)
    print(f"resumed from {RESUME_PATH}: step={start_step}, epoch={start_epoch}")

global_step = start_step
running_loss = 0.0
running_count = 0


def _ckpt(name, slim):
    save_checkpoint(
        f"{CHECKPOINT_DIR}/{name}", model, ema_model, optimizer, lr_scheduler,
        step=global_step, epoch=epoch + 1, loss_history=loss_history,
        scaler=scaler, config=MODEL_CONFIG, slim=slim,
    )


for epoch in range(start_epoch, NUM_EPOCHS):

    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for x_0, context, view in pbar:
        loss = train_step(
            model=model,
            ema_model=ema_model,
            noise_scheduler=noise_scheduler,
            optimizer=optimizer,
            lr_scheduler=lr_scheduler,
            x_0=x_0.to(device),
            context=context.to(device),
            uncond_embedding=uncond_embedding,
            cfg_dropout_prob=CFG_DROPOUT,
            grad_clip=GRAD_CLIP,
            ema_decay=EMA_DECAY,
            min_snr_gamma=MIN_SNR_GAMMA,
            view=view.to(device) if VIEW_COND else None,
            amp_dtype=amp_dtype,
            scaler=scaler,
            step=global_step if EMA_WARMUP else None,
        )

        loss_history.append(loss)
        running_loss += loss
        running_count += 1
        global_step += 1

        if global_step % LOG_EVERY == 0:
            pbar.set_postfix(loss=f"{running_loss / running_count:.4f}",
                             lr=f"{optimizer.param_groups[0]['lr']:.2e}")
            running_loss = 0.0
            running_count = 0

    if (epoch + 1) % VAL_EVERY == 0:
        vl = validation_loss(ema_model, noise_scheduler, val_x0, val_ctx,
                             min_snr_gamma=MIN_SNR_GAMMA, view=val_view)["weighted"]
        tl = validation_loss(ema_model, noise_scheduler, tr_x0, tr_ctx,
                             min_snr_gamma=MIN_SNR_GAMMA, view=tr_view)["weighted"]
        val_history.append((global_step, tl, vl))
        flag = ""
        if KEEP_BEST and vl < best_val:
            best_val, best_epoch = vl, epoch + 1
            _ckpt("best.pt", slim=True)
            flag = "  <-- new best (lowest val MSE, not necessarily best-looking)"
        print(f"epoch {epoch+1:4d}: train {tl:.4f}  val {vl:.4f}  gap {vl - tl:+.4f}{flag}")

    if (epoch + 1) in MILESTONES:
        _ckpt(f"epoch_{epoch+1}.pt", slim=True)
        print(f"  milestone snapshot -> {CHECKPOINT_DIR}/epoch_{epoch+1}.pt")
    if (epoch + 1) % FULL_EVERY == 0 or epoch + 1 == NUM_EPOCHS:
        _ckpt("last.pt", slim=False)

    if PREVIEW and (epoch + 1) % PREVIEW_EVERY == 0:
        make_preview(epoch + 1)

print(f"\nTraining complete. lowest held-out MSE {best_val:.4f} at epoch {best_epoch}")

In [ ]:
losses = np.array(loss_history)
window = max(1, len(losses) // 100)
moving = np.convolve(losses, np.ones(window) / window, mode="valid")

plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.2, label="per-step (train)")
plt.plot(np.arange(window - 1, len(losses)), moving, label=f"moving avg (w={window})")

if val_history:
    vs, tls, vls = zip(*val_history)
    plt.plot(vs, tls, "-", color="darkorange", lw=1, label="train (EMA, eval mode)")
    plt.plot(vs, vls, "o-", color="crimson", ms=3, label="held-out (EMA, eval mode)")
    b = int(np.argmin(vls))
    plt.axvline(vs[b], color="green", ls="--", lw=1, label=f"best val {vls[b]:.3f}")

plt.axhline(1.0, color="grey", ls=":", lw=1, label="predict-zero baseline")
plt.xlabel("step")
plt.ylabel("v-MSE loss")
plt.yscale("log")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

if val_history:
    print(f"best held-out {min(vls):.4f} at step {vs[int(np.argmin(vls))]}, "
          f"final {vls[-1]:.4f}")

## Inference & evaluation

The first cell scores every saved checkpoint, the second loads the one you pick, and
everything after that describes that checkpoint.

**Rerunning after a kernel restart:** these cells also need `DATA` and `caption_lists` from
the dataset cell, `preprocess` from the crop cell, and the VAE and CLIP loader cells. All
four are cheap. Skip only the two encode cells and the uncond-embedding cell.

In [ ]:
# 1. Score every saved checkpoint on train and held-out loss.
import re
from pathlib import Path

from unet import UNet
from scheduler import NoiseScheduler
from train import validation_loss

CKPT_DIR = "checkpoints"

paths = [p for p in Path(CKPT_DIR).glob("*.pt")
         if p.stem in ("best", "last") or re.fullmatch(r"epoch_\d+", p.stem)]
paths.sort(key=lambda p: (p.stem not in ("best", "last"),
                          int(p.stem.split("_")[1]) if "_" in p.stem else 0, p.stem))
if not paths:
    raise FileNotFoundError(f"no .pt checkpoints under {CKPT_DIR}/ — check CKPT_DIR")

# Every checkpoint from one run shares the same schedule, so build it once.
first_cfg = torch.load(paths[0], map_location="cpu").get("config", {})
sweep_sch = NoiseScheduler(
    zero_terminal_snr=first_cfg.get("zero_terminal_snr", True),
    prediction_type=first_cfg.get("prediction_type", "v"),
).to(device)

scored = []
print(f"{'file':>16} {'epoch':>6} {'slim':>5} {'train':>9} {'held-out':>9} {'gap':>9}")
for p in paths:
    ck = torch.load(p, map_location=device)
    c = ck.get("config", {})
    assert c.get("prediction_type", "v") == first_cfg.get("prediction_type", "v"), \
        f"{p.name} uses a different prediction type — not comparable"
    m = UNet(
        base_channels=c.get("base_channels", 64),
        num_res_blocks=c.get("num_res_blocks", 2),
        top_self_attn=c.get("top_self_attn", False),
        dropout=0.0, view_dim=c.get("view_dim", 0),
    ).to(device)
    m.load_state_dict(ck["ema_model"])
    m.eval()

    tl = validation_loss(m, sweep_sch, tr_x0, tr_ctx, view=tr_view)["weighted"]
    vl = validation_loss(m, sweep_sch, val_x0, val_ctx, view=val_view)["weighted"]
    scored.append((p, ck.get("epoch", -1), vl))
    print(f"{p.name:>16} {ck.get('epoch', -1):6d} {str(bool(ck.get('slim'))):>5} "
          f"{tl:9.4f} {vl:9.4f} {vl - tl:+9.4f}")

    del m, ck
    if device == "cuda":
        torch.cuda.empty_cache()

best_path, best_ep, best_vl = min(scored, key=lambda r: r[2])
print(f"\nlowest held-out MSE: {best_path.name} (epoch {best_ep}) at {best_vl:.4f}")
if best_vl > 1.0:
    print("Worse than predicting zero — nothing here generalizes.")
print("Lowest MSE is not always the best-looking — set CKPT_PATH below and compare samples.")

In [ ]:
# 2. Load one checkpoint. Everything below uses eval_ema, which is built here, so after
# changing CKPT_PATH re-run this cell and the ones after it. EVAL_TAG goes on the figures.
import torch.nn.functional as F

from unet import UNet
from scheduler import NoiseScheduler
from sample import sample_ddim, sample_ddpm, latents_to_images

CKPT_PATH = "checkpoints/best.pt"

ckpt = torch.load(CKPT_PATH, map_location=device)
cfg = ckpt.get("config", {})
EVAL_TAG = f"{Path(CKPT_PATH).name} @ epoch {ckpt['epoch']}"
print(f"checkpoint : {CKPT_PATH}")
print(f"epoch      : {ckpt['epoch']}   step: {ckpt['step']}   slim: {bool(ckpt.get('slim'))}")
print(f"config     : {cfg}")

noise_scheduler = NoiseScheduler(
    zero_terminal_snr=cfg.get("zero_terminal_snr", True),
    prediction_type=cfg.get("prediction_type", "v"),
).to(device)
LATENT_SHAPE = tuple(cfg.get("latent_shape", (4, 32, 32)))
CKPT_VIEW_DIM = cfg.get("view_dim", 0)


def build_from_ckpt(state_dict):
    m = UNet(
        base_channels=cfg.get("base_channels", 64),
        num_res_blocks=cfg.get("num_res_blocks", 2),
        top_self_attn=cfg.get("top_self_attn", False),
        dropout=0.0,
        view_dim=CKPT_VIEW_DIM,
    ).to(device)
    m.load_state_dict(state_dict)
    m.eval()
    return m


live_model = build_from_ckpt(ckpt["model"])
eval_ema = build_from_ckpt(ckpt["ema_model"])

live_flat = torch.cat([p.flatten() for p in live_model.parameters()])
ema_flat = torch.cat([p.flatten() for p in eval_ema.parameters()])
rel_delta = (ema_flat - live_flat).norm().item() / live_flat.norm().item()

print(f"\n|ema - live|/|live| = {rel_delta:.4f}   (want << 1; ~1 = EMA still near init)")

In [ ]:
# 3. Loss binned by timestep, train vs held-out.
EVAL_N = 256
N_BINS = 20

cpu_gen = torch.Generator().manual_seed(1234)
tr_pool = train_idx[crop_ids[train_idx] == 0]
va_pool = val_idx[crop_ids[val_idx] == 0]
EVAL_N = min(EVAL_N, len(tr_pool), len(va_pool))

def make_eval_set(pool):
    pick = pool[torch.randperm(len(pool), generator=cpu_gen)[:EVAL_N]]
    x0 = latents_tensor[pick]
    vw = views_for_rows(pick)
    return (x0.to(device), caption_emb_for_rows(pick).to(device),
            torch.randn(x0.shape, generator=cpu_gen).to(device), pick,
            None if vw is None else vw.to(device))

tr_set = make_eval_set(tr_pool)
va_set = make_eval_set(va_pool)
print(f"eval set: {EVAL_N} train / {EVAL_N} held-out latents (crop 0)")

@torch.no_grad()
def loss_at_t(m, t_val, eval_set, ctx=None, batch=32):
    """Mean per-element MSE against the scheduler's target at one fixed timestep."""
    x0, default_ctx, noise, _, vw = eval_set
    ctx = default_ctx if ctx is None else ctx
    total, n = 0.0, 0
    for i in range(0, x0.shape[0], batch):
        xb, cb, nb = x0[i : i + batch], ctx[i : i + batch], noise[i : i + batch]
        vb = None if vw is None else vw[i : i + batch]
        t = torch.full((xb.shape[0],), t_val, device=device, dtype=torch.long)
        x_t = noise_scheduler.q_sample(xb, t, nb)
        target = noise_scheduler.get_target(xb, nb, t)
        pred = m(x_t, t, cb, vb)
        total += F.mse_loss(pred, target, reduction="sum").item()
        n += nb.numel()
    return total / n


# No-text baseline: the lowest loss reachable knowing only each dimension's mean and
# variance, i.e. without reading the caption.
_Xtr = latents_tensor[tr_pool].flatten(1).double()
_mu, _var = _Xtr.mean(0), _Xtr.var(0, unbiased=True)
_s2 = {"train": ((_Xtr - _mu) ** 2).mean(0),
       "val": ((latents_tensor[va_pool].flatten(1).double() - _mu) ** 2).mean(0)}


def no_text_baseline(t_val, which):
    a2 = noise_scheduler.alphas_cumprod[t_val].item(); b2 = 1 - a2
    return ((_s2[which] * b2 + a2 * _var ** 2) / (a2 * _var + b2) ** 2).mean().item()


bin_ts = [int((i + 0.5) * noise_scheduler.T / N_BINS) for i in range(N_BINS)]
tr_curve, va_curve, base_curve = [], [], []

print(f"\n{'t':>5} {'train v':>9} {'val v':>9} {'no-text':>9} {'val vs base':>12}")
for t_val in bin_ts:
    lt = loss_at_t(eval_ema, t_val, tr_set)
    lv = loss_at_t(eval_ema, t_val, va_set)
    bv = no_text_baseline(t_val, "val")
    tr_curve.append(lt); va_curve.append(lv); base_curve.append(bv)
    print(f"{t_val:5d} {lt:9.4f} {lv:9.4f} {bv:9.4f} {1 - lv/bv:+11.1%}")

mt, mv, mb = float(np.mean(tr_curve)), float(np.mean(va_curve)), float(np.mean(base_curve))

plt.figure(figsize=(10, 4))
plt.plot(bin_ts, tr_curve, "o-", label="train (EMA)")
plt.plot(bin_ts, va_curve, "s--", label="held-out (EMA)", alpha=0.8)
plt.plot(bin_ts, base_curve, "-", color="grey", lw=1, label=f"no-text baseline ({mb:.2f})")
plt.axhline(1.0, color="red", ls=":", label="predict-zero")
plt.xlabel("timestep t"); plt.ylabel("v-MSE (eval mode)")
plt.title(f"Denoising loss vs timestep — {EVAL_TAG}")
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f"\nmean   train {mt:.4f}   held-out {mv:.4f}   no-text baseline {mb:.4f}")
print(f"improvement over no-text: train {1-mt/mb:+.1%}   held-out {1-mv/mb:+.1%}")
print(f"at t={bin_ts[-1]}: held-out {1-va_curve[-1]/base_curve[-1]:+.1%}")

In [ ]:
# 4. Is the model using the text, and how?
#
# Same latents, same noise, same t; only the caption changes.
#   matched  = each latent with its own caption
#   shuffled = each latent with someone else's caption
#   uncond   = the "" embedding
#   alt      = a different human caption of the same image (paraphrase robustness)
#
# Run on both sets: a gap on training images can be a memorized caption->image pairing,
# while a gap on held-out images means real conditioning.

shuf = torch.randperm(EVAL_N, generator=torch.Generator().manual_seed(7))
assert (shuf == torch.arange(EVAL_N)).sum().item() < EVAL_N * 0.05

has_alt = stats["captions_per_image"] > 1
uncond_ctx = uncond_embedding.to(device).expand(EVAL_N, -1, -1)


def probe(eval_set, label):
    """Largest relative shuffled-minus-matched gap over a spread of timesteps."""
    rows, matched_ctx = eval_set[3], eval_set[1]
    shuffled_ctx = matched_ctx[shuf.to(matched_ctx.device)]
    alt_ctx = caption_emb_for_rows(rows, variant=1).to(device) if has_alt else None

    hdr = f"{'t':>5}  {'matched':>9}  {'shuffled':>9}  {'uncond':>9}"
    if has_alt:
        hdr += f"  {'alt':>9}"
    hdr += f"  {'shuf rel':>9}"
    print(f"\n--- {label} ---")
    print(hdr)

    gaps = []
    for t_val in [50, 150, 300, 500, 700, 900]:
        l_m = loss_at_t(eval_ema, t_val, eval_set)
        l_s = loss_at_t(eval_ema, t_val, eval_set, ctx=shuffled_ctx)
        l_u = loss_at_t(eval_ema, t_val, eval_set, ctx=uncond_ctx)
        gaps.append((l_s - l_m) / max(l_m, 1e-12))
        line = f"{t_val:5d}  {l_m:9.4f}  {l_s:9.4f}  {l_u:9.4f}"
        if has_alt:
            line += f"  {loss_at_t(eval_ema, t_val, eval_set, ctx=alt_ctx):9.4f}"
        line += f"  {gaps[-1]:8.1%}"
        print(line)
    return max(gaps)


tr_gap = probe(tr_set, "training images")
va_gap = probe(va_set, "held-out images")

print(f"\nlargest relative shuffled-minus-matched gap")
print(f"  training  {tr_gap:8.1%}")
print(f"  held-out  {va_gap:8.1%}   <- the one that matters")

if va_gap < 0.01:
    print("\nThe caption does not change the prediction on unseen images.")
elif va_gap < 0.05:
    print("\nWeak conditioning on unseen images.")
elif tr_gap > 5.0 and va_gap < tr_gap / 5:
    print("\nLarge gap on training but not held-out: caption->image lookup, not understanding.")
else:
    print("\nConditioning is in good range.")

In [ ]:
# 5. Helpers: encode prompts, generate, show a row, check latent statistics.
CFG = 2.0
CFG_RESCALE = 0.7

# Clamp on each x_0 estimate while sampling: the 99.99th percentile of |x_0| in the
# training latents.
_a = latents_tensor.abs().flatten()
CLIP_X0 = round(_a.kthvalue(int(0.9999 * _a.numel())).values.item(), 1)
print(f"CLIP_X0 = {CLIP_X0}   (99.99th pct of |x_0|; abs max {_a.max().item():.1f})")
del _a

@torch.no_grad()
def encode_prompt(prompts):
    """list[str] -> (B, MAX_LENGTH, 768) CLIP embeddings, same pipeline as training."""
    if isinstance(prompts, str):
        prompts = [prompts]
    ids = tokenizer(
        prompts, padding="max_length", max_length=MAX_LENGTH, truncation=True, return_tensors="pt",
    ).input_ids.to(device)
    return text_encoder(ids).last_hidden_state


@torch.no_grad()
def generate(cond_emb, method="ddim", num_steps=50, guidance_scale=None,
             guidance_rescale=None, eta=0.0, seed=42, fixed_noise=False, view=None):
    """
    Sample latents with the EMA model. Returns (images in [0,1], raw latents).

    view defaults to the canonical full-frame unflipped view.
    fixed_noise=True gives every row the same starting noise — use it when comparing
    prompts, otherwise the difference is mostly the noise draw.
    """
    gs = CFG if guidance_scale is None else guidance_scale
    gr = CFG_RESCALE if guidance_rescale is None else guidance_rescale
    g = torch.Generator(device=device).manual_seed(seed)

    B = cond_emb.shape[0]
    x_T = None
    if fixed_noise:
        x_T = torch.randn(1, *LATENT_SHAPE, device=device, generator=g).expand(B, -1, -1, -1).contiguous()
    if view is None and CKPT_VIEW_DIM > 0:
        view = canonical_views(B).to(device)

    kw = dict(guidance_scale=gs, guidance_rescale=gr, num_steps=num_steps,
              latent_shape=LATENT_SHAPE, clip_x0=CLIP_X0, x_T=x_T, view=view, generator=g)
    if method == "ddim":
        lat = sample_ddim(eval_ema, noise_scheduler, cond_emb,
                          uncond_embedding.to(device), eta=eta, **kw)
    else:
        lat = sample_ddpm(eval_ema, noise_scheduler, cond_emb,
                          uncond_embedding.to(device), **kw)

    imgs = latents_to_images(lat, vae, lat_mean=LAT_MEAN, lat_std=LAT_STD).cpu()
    return imgs, lat


def show_row(images, titles, suptitle=None, wrap=28):
    """images: (B, 3, H, W) in [0, 1]."""
    n = images.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.8))
    axes = [axes] if n == 1 else list(axes)
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
        wrapped = "\n".join(str(title)[i : i + wrap] for i in range(0, min(len(str(title)), wrap * 3), wrap))
        ax.set_title(wrapped, fontsize=8)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    plt.tight_layout()
    plt.show()

_REF_SAMPLE_STD = latents_tensor.flatten(1).std(dim=1).mean().item()
_REF_MEAN_SPREAD = latents_tensor.flatten(1).mean(dim=1).std().item()


def latent_stats(name, lat):
    """Compare generated latent statistics against the training latents."""
    flat = lat.flatten(1)
    per_sample = flat.std(dim=1).mean().item()
    pc_mean = lat.mean(dim=(0, 2, 3)).tolist()
    line = (f"{name:20s} per-sample std {per_sample:.3f} "
            f"(ref {_REF_SAMPLE_STD:.3f}, {per_sample / _REF_SAMPLE_STD - 1:+.0%})")
    if lat.shape[0] > 1:
        spread = flat.mean(dim=1).std().item()
        line += (f"  mean-spread {spread:.3f} (ref {_REF_MEAN_SPREAD:.3f}, "
                 f"{spread / max(_REF_MEAN_SPREAD, 1e-9):.1f}x)")
    line += f"  per-chan mean {[round(v, 2) for v in pc_mean]}"
    print(line)


print(f"TRAINING reference   per-chan mean={[round(v, 3) for v in latents_tensor.mean(dim=(0,2,3)).tolist()]}")
print(f"                     per-sample std {_REF_SAMPLE_STD:.3f}   "
      f"mean-spread {_REF_MEAN_SPREAD:.3f}")

In [ ]:
# 6. Samples for captions the model trained on, next to the real images.
captions = [caption_lists[i][0] for i in SHOW_IMAGES]
cond = caption_emb(SHOW_IMAGES).to(device)

imgs, lats = generate(cond, method="ddim", num_steps=50, seed=SEED)
latent_stats(f"generated (cfg {CFG})", lats)
show_row(imgs, captions, suptitle=f"{EVAL_TAG} — DDIM 50, CFG {CFG} (rescale {CFG_RESCALE}), training captions")

real = torch.stack([preprocess(DATA[i][IMAGE_COL]) for i in SHOW_IMAGES])
show_row((real + 1) / 2, captions, suptitle="Ground truth for the same captions")

In [ ]:
# 7. Guidance sweep: same prompt and seed at several CFG scales.
CFG_IMAGE = SHOW_IMAGES[0]
cfg_caption = caption_lists[CFG_IMAGE][0]
cfg_cond = caption_emb([CFG_IMAGE]).to(device)

print(f"prompt: {cfg_caption}\n")
scales = [1.5, 2.0, 3.0, 5.0]

cfg_imgs = []
for w in scales:
    img, lat = generate(cfg_cond, method="ddim", num_steps=50, guidance_scale=w,
                        guidance_rescale=CFG_RESCALE, seed=SEED)
    latent_stats(f"cfg {w}", lat)
    cfg_imgs.append(img[0])

show_row(torch.stack(cfg_imgs), [f"CFG {w}" for w in scales],
         suptitle=f"{EVAL_TAG} — guidance sweep at rescale {CFG_RESCALE}, same seed")

In [ ]:
# 8. Sampler comparison: step count and stochasticity, same seed.
sweep_cond = caption_emb([CFG_IMAGE]).to(device)

variants = [
    ("ddim", 25, 0.0), ("ddim", 50, 0.0), ("ddim", 250, 0.0), ("ddim", 50, 1.0),
]

sweep_imgs, sweep_titles = [], []
for method, steps, eta in variants:
    img, lat = generate(sweep_cond, method=method, num_steps=steps, eta=eta, seed=SEED)
    label = f"{method} {steps}" + (f" eta{eta}" if eta else "")
    latent_stats(label, lat)
    sweep_imgs.append(img[0])
    sweep_titles.append(label.upper())

show_row(torch.stack(sweep_imgs), sweep_titles,
         suptitle=f"{EVAL_TAG} — sampler comparison (CFG {CFG}, same seed)")

In [ ]:
# 9. Prompts the model has never seen, sharing one starting noise.
novel = [
    "a smiling young person with long blond hair and high cheekbones",
    "an older person with a grey beard, glasses and receding hair",
    "a young person with short black hair, arched eyebrows, not smiling",
    "a person with dark curly hair, heavy makeup and wearing earrings",
]
novel_cond = encode_prompt(novel)

novel_imgs, novel_lats = generate(novel_cond, method="ddim", num_steps=50,
                                  seed=SEED, fixed_noise=True)
latent_stats("generated (novel)", novel_lats)
show_row(novel_imgs, novel, suptitle=f"Unseen prompts, shared starting noise (DDIM 50, CFG {CFG})")

flat = novel_lats.flatten(1)
dist = torch.cdist(flat, flat).cpu()
print("\npairwise latent L2 between samples (same x_T, so this is prompt effect only):")
print(dist.numpy().round(2))
off = dist[~torch.eye(len(novel), dtype=bool)]
print(f"mean off-diagonal: {off.mean().item():.2f}")

_, indep_lats = generate(novel_cond, method="ddim", num_steps=50, seed=SEED, fixed_noise=False)
iflat = indep_lats.flatten(1)
idist = torch.cdist(iflat, iflat).cpu()
ioff = idist[~torch.eye(len(novel), dtype=bool)]
print(f"same prompts, independent noise: mean off-diagonal {ioff.mean().item():.2f}")
print(f"prompt effect / total variation : {off.mean().item() / ioff.mean().item():.1%}")

In [ ]:
# 10. Memorization: is a sample closer to a training image than to a held-out one?
MEM_REF_N = 4000        # rows used for the train-vs-train reference distribution

mem_rows = train_idx[crop_ids[train_idx] == 0]
train_flat = latents_tensor[mem_rows].flatten(1)

# This distribution is O(n^2) in memory. Run 5 died here: all 54,000 crop-0 rows is an
# 11.7 GB distance matrix plus an 11.7 GB masked copy, which OOM-killed the instance. And
# torch.quantile caps at 2^24 elements, the bug that killed run 3's eval. Estimate the
# percentiles from a subsample; the nearest-neighbour search below still uses every row.
_g = torch.Generator().manual_seed(0)
n_ref = min(MEM_REF_N, len(mem_rows))
ref_flat = train_flat[torch.randperm(len(mem_rows), generator=_g)[:n_ref]]
off_diag = torch.cdist(ref_flat, ref_flat)[~torch.eye(n_ref, dtype=torch.bool)]

P1 = off_diag.kthvalue(max(1, int(0.01 * off_diag.numel()))).values.item()
MINDIST = off_diag.min().item()
MEDIAN = off_diag.median().item()

print(f"train-vs-train latent L2 ({n_ref} of {len(mem_rows)} crop-0 rows): "
      f"mean {off_diag.mean():.2f}  median {MEDIAN:.2f}  p1 {P1:.2f}  min {MINDIST:.2f}")
print(f"  -> suspicious below {P1:.2f}, unambiguous copy below {MINDIST:.2f}\n")

gen_imgs, gen_lats = generate(caption_emb(SHOW_IMAGES).to(device), num_steps=50, seed=SEED)
d = torch.cdist(gen_lats.flatten(1).cpu(), train_flat)
nn_dist, nn_pos = d.min(dim=1)
nn_rows = mem_rows[nn_pos]

for i, (row, dist_i) in enumerate(zip(nn_rows.tolist(), nn_dist.tolist())):
    img_id = int(group_ids[row])
    tag = " (mirrored)" if int(stats["flip_ids"][row]) else ""
    flag = "  <-- COPY" if dist_i < MINDIST else ("  <-- suspicious, inspect" if dist_i < P1 else "")
    print(f"sample {i}: nearest = image {img_id:3d}{tag}  dist {dist_i:7.2f}  "
          f"({dist_i / MEDIAN:.2f}x median){flag}")

val_flat = latents_tensor[val_idx[crop_ids[val_idx] == 0]].flatten(1)
d_val = torch.cdist(gen_lats.flatten(1).cpu(), val_flat).min(dim=1).values
print(f"\nmean nearest to TRAIN {nn_dist.mean():.2f}   to HELD-OUT {d_val.mean():.2f}"
      f"   ratio {nn_dist.mean() / d_val.mean():.3f}  (want ~1.0)")

nn_real = []
for row in nn_rows.tolist():
    im = preprocess(DATA[int(group_ids[row])][IMAGE_COL])
    nn_real.append(torch.flip(im, dims=[-1]) if int(stats["flip_ids"][row]) else im)
show_row(gen_imgs, [f"generated {i}" for i in range(len(nn_rows))], suptitle="Generated")
show_row((torch.stack(nn_real) + 1) / 2,
         [f"nearest image #{int(group_ids[r])}" for r in nn_rows.tolist()],
         suptitle="Nearest training latent (crop 0)")